# 01 - Data preparation
Runs the data-preparation pipeline for the machine translation project:
- use corpus from OPUS/Tatoeba (in data/raw/    link:https://opus.nlpl.eu/datasets/Tatoeba?pair=en&fr) (can use a different dataset later)
- Tokenization, vocabulary construction
- Encoding sentence pairs, train/val/test split
- Save processed files to `data/processed/` and create a `DataLoader` sample

In [1]:
import os
import sys
from pprint import pprint

sys.path.append(os.path.abspath(os.path.join('..','src')))

import torch
from torch.utils.data import DataLoader

import dataset
from dataset import prepare_parallel_data, TranslationDataset, collate_fn
from vocabulary import Vocabulary

print('torch:', torch.__version__)
print('dataset module:', dataset.__file__)

torch: 2.8.0+cu128
dataset module: /home/jovyan/Machine_Translation/src/dataset.py


## Configuration
Set paths and options. This notebook expects the parallel corpus to be prepared manually in `data/raw/src.txt` and `data/raw/tgt.txt`.

In [2]:
src_lang = 'en'
tgt_lang = 'fr'
raw_src = os.path.abspath(os.path.join('..','data','raw','src.txt'))
raw_tgt = os.path.abspath(os.path.join('..','data','raw','tgt.txt'))
outdir = os.path.abspath(os.path.join('..','data','processed'))
min_freq = 1

print('raw_src ->', raw_src)
print('raw_tgt ->', raw_tgt)

# Ensure folders exist
os.makedirs(os.path.dirname(raw_src), exist_ok=True)
os.makedirs(os.path.dirname(outdir), exist_ok=True)

if not (os.path.exists(raw_src) and os.path.exists(raw_tgt)):
    raise FileNotFoundError(
        'Missing raw data files. Please place the parallel corpus manually at data/raw/src.txt and data/raw/tgt.txt before running this notebook.'
    )

print('Raw files found locally.')

raw_src -> /home/jovyan/Machine_Translation/data/raw/src.txt
raw_tgt -> /home/jovyan/Machine_Translation/data/raw/tgt.txt
Raw files found locally.


In [3]:
# Run the preprocessing pipeline (build vocabs, encode, split, save)
# This can take some time depending on corpus size.
if not (os.path.exists(raw_src) and os.path.exists(raw_tgt)):
    raise RuntimeError('Raw data not available. Place files in data/raw/ or re-run previous cell with a successful download.')

src_vocab, tgt_vocab, train, val, test = prepare_parallel_data(raw_src, raw_tgt, outdir=outdir, min_freq=min_freq, save=True)

print('Vocab sizes:')
print('  src:', len(src_vocab))
print('  tgt:', len(tgt_vocab))
print()
print('Dataset sizes:')
print('  train:', len(train))
print('  val:  ', len(val))
print('  test: ', len(test))

# show a decoded example (strip special tokens for readability)
def strip_special(tokens):
    return [t for t in tokens if t not in (Vocabulary.PAD, Vocabulary.SOS, Vocabulary.EOS, Vocabulary.UNK)]

if len(train) > 0:
    s_ids, t_ids = train[0]
    print('\nSample pair (first from train):')
    print('  src ids:', s_ids[:20])
    print('  tgt ids:', t_ids[:20])
    print('  src tokens:', ' '.join(strip_special(src_vocab.decode(s_ids))))
    print('  tgt tokens:', ' '.join(strip_special(tgt_vocab.decode(t_ids))))

Vocab sizes:
  src: 29961
  tgt: 48596

Dataset sizes:
  train: 246491
  val:   13693
  test:  13695

Sample pair (first from train):
  src ids: [1, 47, 9286, 61, 5, 473, 4, 2]
  tgt ids: [1, 64, 12477, 38, 70, 4, 2]
  src tokens: they chatted about the weather .
  tgt tokens: ils discutèrent du temps .


In [4]:
# Create a DataLoader sample to verify collate_fn and batching
from torch.utils.data import DataLoader

train_ds = TranslationDataset(train)
pad_id = src_vocab.token2id[Vocabulary.PAD]
loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=lambda b: collate_fn(b, pad_id=pad_id))

batch = next(iter(loader))
src_batch, src_lens, tgt_batch, tgt_lens = batch
print('Batch shapes:')
print('  src:', src_batch.shape)
print('  src_lens:', src_lens)
print('  tgt:', tgt_batch.shape)
print('  tgt_lens:', tgt_lens)

Batch shapes:
  src: torch.Size([8, 13])
  src_lens: tensor([ 9,  8,  9, 13,  9,  7,  7,  6])
  tgt: torch.Size([8, 11])
  tgt_lens: tensor([ 8,  7,  8,  9, 10,  8, 11,  6])
